## Nota de revisión

Caso académico, no un trabajo comercial acreditado para ConnectaTel. Las cifras e interpretaciones heredadas describen la ejecución original y **no están revalidadas**: faltan los CSV originales. Las correcciones de filtrado anual y minutos pueden cambiar los resultados. Se eliminaron salidas antiguas para evitar presentarlas como resultados del código actualizado. No se publicaron datos de curso ni datos de clientes.


# Análisis ConnectaTel

Como **analista de datos**, tu objetivo es evaluar el **comportamiento de los clientes** de una empresa de telecomunicaciones en Latinoamérica, ConnectaTel. 

Trabajaremos con información registrada **hasta el año 2024**, lo cual permitirá analizar el comportamiento del negocio dentro de ese periodo.

Para ello trabajarás con tres datasets:  

- **plans.csv** → información de los planes actuales (precio, minutos incluidos, GB incluidos, costo por extra)  
- **users.csv** → información de los clientes (edad, ciudad, fecha de registro, plan, churn)  
- **usage.csv** → detalle del **uso real** de los servicios (llamadas y mensajes)  

Deberás **explorar**, **limpiar** y **analizar** estos datos para construir un **perfil estadístico** de los clientes, detectar **comportamientos atípicos** y crear **segmentos de clientes**.  

Este análisis te permitirá **identificar patrones de consumo**, **diseñar estrategias de retención** y **sugerir mejoras en los planes** ofrecidos por la empresa.

> 💡 Antes de empezar, recuerda pensar de forma **programática**: ¿qué pasos necesitas? ¿En qué orden? ¿Qué quieres medir y por qué?


--- 
## 🧩 Paso 1: Cargar y explorar

Antes de limpiar o combinar los datos, es necesario **familiarizarte con la estructura de los tres datasets**.  
En esta etapa, validarás que los archivos se carguen correctamente, conocerás sus columnas y tipos de datos, y detectarás posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:**  
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

**Instrucciones:**  
- Importa las librerías necesarias (por ejemplo `pandas`, `seaborn`, `matplotlib.pyplot`)
- Carga los archivos CSV usando `pd.read_csv()`:
  - **`/datasets/plans.csv`**  
  - **`/datasets/users_latam.csv`**  
  - **`/datasets/usage.csv`**  
- Guarda los DataFrames en las variables: `plans`, `users`, `usage`.  
- Muestra las primeras filas de cada DataFrame usando `.head()`.


In [ ]:
# importar librerías
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Resolver los datos desde la raíz del repositorio o desde notebooks/.
from pathlib import Path
import os

required_files = ('plans.csv', 'users_latam.csv', 'usage.csv')
if os.environ.get('CONNECTATEL_DATA_DIR'):
    DATA_DIR = Path(os.environ['CONNECTATEL_DATA_DIR']).expanduser().resolve()
else:
    root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / 'notebooks' / 'connectatel_analysis.ipynb').is_file()), None)
    DATA_DIR = root / 'data' / 'raw' if root else Path('/datasets')
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Faltan {missing} en {DATA_DIR}. Consulta data/raw/README.md; "
                            "puedes indicar una carpeta autorizada con CONNECTATEL_DATA_DIR.")
plans = pd.read_csv(DATA_DIR / 'plans.csv')
users = pd.read_csv(DATA_DIR / 'users_latam.csv')
usage = pd.read_csv(DATA_DIR / 'usage.csv')
expected = [(users, {'user_id', 'age', 'city', 'plan', 'reg_date', 'churn_date'}),
            (usage, {'id', 'user_id', 'type', 'date', 'duration', 'length'})]
for frame, columns in expected:
    if columns - set(frame.columns):
        raise ValueError(f"Columnas faltantes: {sorted(columns - set(frame.columns))}")
if users['user_id'].isna().any() or users['user_id'].duplicated().any():
    raise ValueError('users.user_id debe ser único y no nulo.')
if usage['id'].isna().any() or usage['id'].duplicated().any():
    raise ValueError('usage.id debe ser único y no nulo; revisar duplicados antes de agregar.')
if not usage['user_id'].isin(users['user_id']).all():
    raise ValueError('Existen eventos sin un usuario válido.')
if not usage['type'].isin(['call', 'text']).all():
    raise ValueError('Tipos de evento no admitidos; revisar antes de agregar.')


In [ ]:
# mostrar las primeras 5 filas de plans
plans.head()

In [ ]:
# mostrar las primeras 5 filas de users
users.head()

In [ ]:
# mostrar las primeras 5 filas de usage
usage.head()

**Tip:** Si no usas `print()` la tabla se vera mejor.

### 1.2 Exploración de la estructura de los datasets

**🎯 Objetivo:**  
Conocer la **estructura de cada dataset**, revisar cuántas filas y columnas tienen, identificar los **tipos de datos** de cada columna y detectar posibles **inconsistencias o valores nulos** antes de iniciar el análisis.

**Instrucciones:**  
- Revisa el **número de filas y columnas** de cada dataset usando `.shape`.  
- Usa `.info()` en cada DataFrame para obtener un **resumen completo** de columnas, tipos de datos y valores no nulos.  

In [ ]:
# revisar el número de filas y columnas de cada dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

In [ ]:
# inspección de plans con .info()
plans.info()

In [ ]:
# inspección de users con .info()
users.info()

In [ ]:
# inspección de usage con .info()
usage.info()

---

## 🧩Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos

**🎯 Objetivo:**  
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

**Instrucciones:**  
- Cuenta valores nulos por columna para cada dataset.
- Calcula la proporción de nulos por columna para cada dataset.

El dataset `plans` solamente tiene 2 renglones y se puede observar que no tiene ausentes, por ello no necesita exploración adicional.

<br>
<details>
<summary>Haz clic para ver la pista</summary>
Usa `.isna().sum()` para contar valores nulos y usa `.isna().mean()` para calcular la proporción.

In [ ]:
# cantidad de nulos para users
print("# Cantidad de valores nulos")
print(users.isna().sum())

print("# Proporción de valores nulos")
print(users.isna().mean())

In [ ]:
# cantidad de nulos para usage
print("# Cantidad de valores nulos")
print(usage.isna().sum())

print("# Proporción de valores nulos")
print(usage.isna().mean())

✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico al final del bloque. Incluye qué ves y que acción recomendarías para cada caso.

💡 **Nota:** Justifica tus decisiones brevemente (1 línea por caso).
* Hint:
 - Si una columna tiene **más del 80–90% de nulos**, normalmente se **ignora o elimina**.  
 - Si tiene **entre 5% y 30%**, generalmente se **investiga para imputar o dejar como nulos**.  
 - Si es **menor al 5%**, suele ser un caso simple de imputación o dejar como nulos. 
 
 ---


**Valores nulos**

En el dataset `users`, las columnas con valores faltantes son `city` y `churn_date`.  
La columna `city` tiene 469 valores nulos, equivalentes al 11.72% del total. En este caso, se recomienda investigar el origen de los datos faltantes y, si no es posible recuperarlos, imputar con una categoría como `"Unknown"` para no perder registros de clientes.

La columna `churn_date` tiene 3534 valores nulos, equivalentes al 88.35% del total. En este caso, no se debe eliminar ni imputar directamente, ya que probablemente los valores nulos indican clientes activos que no han cancelado el servicio.

En el dataset `usage`, la columna `date` tiene 50 valores nulos, equivalentes al 0.12% del total. Al ser una proporción muy baja, se podrían eliminar esos registros o dejarlos fuera de análisis temporales.

Las columnas `duration` y `length` también presentan valores nulos: `duration` tiene 22076 nulos, equivalentes al 55.19%, y `length` tiene 17896 nulos, equivalentes al 44.74%. Sin embargo, estos nulos son esperados porque `duration` aplica para llamadas y `length` aplica para mensajes. Por lo tanto, no se deben eliminar automáticamente; deben tratarse según el tipo de registro (`call` o `text`).

### 2.2 Detección de valores inválidos y sentinels

🎯 **Objetivo:**  
Identificar sentinels: valores que no deberían estar en el dataset.

**Instrucciones:**
- Explora las columnas numéricas con **un resumen estadístico** y describe brevemente que encontraste.
- Explora las columnas categóricas **relevantes**, revisando sus valores únicos y describe brevemente que encontraste.


El dataset `plans` solamente tiene 2 renglones, por ello no necesita exploración adicional.

In [ ]:
# explorar columnas numéricas de users
users[['user_id', 'age']].describe()

- La columna `user_id` no presenta valores inválidos aparentes. Tiene 4000 registros, con IDs entre 10000 y 13999, lo cual parece consistente para un identificador de usuario.
- La columna `age` presenta un valor inválido o sentinel, ya que el mínimo es -999. Esta edad no es lógica, por lo tanto debe reemplazarse por NaN y luego decidir si se imputa o se excluye según el análisis.

In [ ]:
# explorar columnas numéricas de usage
usage[['id', 'user_id', 'duration', 'length']].describe()

- Las columnas `id` y `user_id` no presentan valores inválidos aparentes. Sus rangos parecen consistentes con identificadores.
- En la columna `duration`, el mínimo es 0 y el máximo es 120. Se debe revisar si duración igual a 0 representa una llamada no contestada, error de registro o valor válido del negocio.
- En la columna `length`, el mínimo es 0 y el máximo es 1490. Se debe revisar si mensajes con longitud 0 son válidos o representan errores. El valor máximo también podría revisarse como posible outlier.

In [ ]:
# explorar columnas categóricas de users
columnas_user = ['city', 'plan']

users[columnas_user].describe()

- La columna `city` tiene 3531 registros no nulos de 4000, por lo tanto presenta valores faltantes. Tiene 7 ciudades únicas y la ciudad más frecuente es Bogotá con 808 registros. Se recomienda imputar los valores faltantes como "Unknown" o investigar su origen.
- La columna `plan` no presenta valores faltantes. Tiene 2 categorías únicas y el plan más frecuente es Básico con 2595 registros.

In [ ]:
# explorar columna categórica de usage
usage['type'].value_counts()

- La columna `type` tiene dos categorías: `text` y `call`. Hay 22092 registros de mensajes y 17908 registros de llamadas. No se observan categorías inválidas aparentes.

### Valores inválidos o sentinels

Se identificó un valor inválido en la columna `age`, ya que el mínimo es -999. Este valor no representa una edad real, por lo tanto se recomienda reemplazarlo por NaN y posteriormente imputarlo o tratarlo como dato faltante.

En `usage`, las columnas `duration` y `length` presentan valores mínimos de 0. Estos valores deben revisarse según la lógica del negocio, ya que podrían representar llamadas sin duración, mensajes vacíos o errores de registro.

También se observa que `length` tiene un valor máximo de 1490, que podría ser un posible outlier en la longitud de mensajes y debería analizarse con histogramas o boxplots.

No se observan valores inválidos aparentes en `plan` ni en `type`, ya que sus categorías son consistentes con el negocio.

### 2.3 Revisión y estandarización de fechas

**🎯 Objetivo:**  
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

**Instrucciones:**  
- Convierte las columnas de fecha a tipo fecha y asegurate de que el código sea a prueba de errores.  
- Revisa cuántas veces aparece cada año.
- Identifica fechas imposibles (ej. años futuros o negativos).

Toma en cuenta que tenemos datos registrados hasta el año 2024.

In [ ]:
# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'], errors='coerce')

In [ ]:
# Convertir a fecha la columna `date` de usage
usage['date'] = pd.to_datetime(usage['date'], errors='coerce')

In [ ]:
# Revisar los años presentes en `reg_date` de users
users['reg_date'].dt.year.value_counts().sort_index()


Los recuentos de años se muestran en la celda anterior. Las fechas posteriores a 2024 se marcarán como inválidas.


In [ ]:
# Revisar los años presentes en `date` de usage
usage['date'].dt.year.value_counts().sort_index()

Los eventos se restringen a 2024 antes de agregar; los eventos sin fecha se reportan y excluyen del perfil anual.


### Fechas fuera de rango

Se convirtieron las columnas `reg_date` de `users` y `date` de `usage` a formato fecha usando `pd.to_datetime()` con `errors='coerce'`, lo que permite transformar fechas inválidas en `NaT`.

En `reg_date`, se encontraron registros en 2022, 2023, 2024 y 2026. Como el análisis solo considera datos hasta el año 2024, los 40 registros del año 2026 deben revisarse como posibles errores de captura. Se recomienda corregirlos si existe una fuente confiable o excluirlos del análisis.

En `date` de `usage`, los registros con fecha válida corresponden al año 2024, por lo que no se observan años fuera de rango. Sin embargo, existen 50 registros sin fecha válida, los cuales podrían excluirse de análisis temporales debido a que representan una proporción muy baja del dataset.

---

## 🧩Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

**Instrucciones:**  
- En `age`, reemplaza el sentinel **-999** con la mediana.
- En `city`, reemplaza el sentinel `"?"` por valores nulos (`pd.NA`).  
- Marca como nulas (`pd.NA`) las fechas fuera de rango.

In [ ]:
# Reemplazar -999 por la mediana de age
age_mediana = users.loc[users['age'] != -999, 'age'].median()
users['age'] = users['age'].replace(-999, age_mediana)

# Verificar cambios
users['age'].describe()

In [ ]:
# Reemplazar ? por NA en city
users['city'] = users['city'].replace('?', pd.NA)

# Verificar cambios
users['city'].value_counts(dropna=False)

In [ ]:
# Marcar fechas futuras como NA para reg_date
users.loc[users['reg_date'].dt.year > 2024, 'reg_date'] = pd.NaT

# Verificar cambios
users['reg_date'].dt.year.value_counts(dropna=False).sort_index()

### Nulos estructurales por tipo de evento

Una duración no aplica a mensajes y una longitud no aplica a llamadas. La tabla cruzada permite revisar inconsistencias; por sí sola no demuestra un mecanismo estadístico MAR.


In [ ]:
# Verificación MAR en usage (Missing At Random) para duration
pd.crosstab(usage['type'], usage['duration'].isna())

In [ ]:
# Verificación MAR en usage (Missing At Random) para length
pd.crosstab(usage['type'], usage['length'].isna())

### Diagnóstico de nulos en `duration` y `length`

Los valores nulos en `duration` y `length` dependen principalmente de la columna `type`, por lo tanto no parecen ser nulos completamente aleatorios.

En `duration`, todos los registros de tipo `call` tienen duración registrada: 17.908 llamadas tienen valor en `duration` y 0 llamadas tienen `duration` nulo. En cambio, la mayoría de registros de tipo `text` tienen `duration` nulo: 22.076 mensajes no tienen duración, lo cual es esperado porque los mensajes no tienen duración de llamada. Sin embargo, hay 16 registros de tipo `text` con valor en `duration`, por lo que deben revisarse como posibles inconsistencias.

En `length`, todos los registros de tipo `text` tienen longitud registrada: 22.092 mensajes tienen valor en `length` y 0 mensajes tienen `length` nulo. En cambio, la mayoría de registros de tipo `call` tienen `length` nulo: 17.896 llamadas no tienen longitud de mensaje, lo cual es esperado. Sin embargo, hay 12 registros de tipo `call` con valor en `length`, por lo que deben revisarse como posibles inconsistencias.

La recomendación es no imputar de forma general los nulos de `duration` y `length`, ya que son nulos esperados según el tipo de registro. Para el análisis de llamadas se debe usar `duration`, y para el análisis de mensajes se debe usar `length`. Los 16 textos con duración y las 12 llamadas con longitud deben revisarse o marcarse como inconsistencias antes del análisis final.

---

## 🧩Paso 4: Summary statistics de uso por usuario


### 4.1 Agrupación por comportamiento de uso

🎯**Objetivo**: Resumir las variables clave de la tabla `usage` **por usuario**, creando métricas que representen su comportamiento real de uso histórico. 

**Instrucciones:**: 
1. Construye una tabla agregada de `usage` por `user_id` que incluya:
- número total de mensajes  
- número total de llamadas  
- total de minutos de llamadas

2. Renombra las columnas para que tengan nombres claros:  
- `cant_mensajes`  
- `cant_llamadas`  
- `cant_minutos_llamada`
3. Combina esta tabla con `users`.

In [ ]:
# Solo eventos fechados en 2024; no sumar duraciones de mensajes.
usage_2024 = usage.loc[usage['date'].dt.year.eq(2024)].copy()
print('Eventos fuera de 2024 o sin fecha excluidos:', len(usage) - len(usage_2024))
usage_2024['is_text'] = usage_2024['type'].eq('text').astype(int)
usage_2024['is_call'] = usage_2024['type'].eq('call').astype(int)
usage_2024['call_minutes'] = pd.to_numeric(usage_2024['duration'], errors='coerce').where(usage_2024['type'].eq('call'), 0)
if (usage_2024.loc[usage_2024['type'].eq('call'), 'call_minutes'].isna().any()
        or usage_2024['call_minutes'].lt(0).any()):
    raise ValueError('Hay llamadas sin duración válida; revisar antes de sumar.')
usage_agg = usage_2024.groupby('user_id').agg(
    is_text=('is_text', 'sum'), is_call=('is_call', 'sum'),
    duration=('call_minutes', 'sum')).reset_index()
usage_agg.head(3)


In [ ]:
# Renombrar columnas
usage_agg = usage_agg.rename(columns={
    "is_text": "cant_mensajes",
    "is_call": "cant_llamadas",
    "duration": "cant_minutos_llamada"
})

# observar resultado
usage_agg.head(3)

In [ ]:
user_profile = users.merge(usage_agg, on='user_id', how='left', validate='one_to_one')
metrics = ['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']
user_profile[metrics] = user_profile[metrics].fillna(0)
# Cero significa sin eventos observados en 2024, no prueba de inactividad real.
user_profile.head(5)


### 4.2 4.2 Resumen estadístico por usuario durante el 2024

🎯 **Objetivo:** Analizar las columnas numéricas y categóricas de los usuarios, para identificar rangos, valores extremos y distribución de los datos antes de continuar con el análisis.

**Instrucciones:**  
1. Para las columnas **numéricas** relevantes, obtén un resumen estadístico (media, mediana, mínimo, máximo, etc.).  
2. Para la columna **categórica** `plan`, revisa la distribución en **porcentajes** de cada categoría.

In [ ]:
# Resumen estadístico de las columnas numéricas
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

user_profile[columnas_numericas].describe()

In [ ]:
# Distribución porcentual del tipo de plan
user_profile['plan'].value_counts(normalize=True) * 100

---

## 🧩Paso 5: Visualización de distribuciones (uso y clientes) y outliers


### 5.1 Visualización de Distribuciones

🎯 **Objetivo:**  
Entender visualmente cómo se comportan las variables clave tanto de **uso** como de **clientes**, observar si existen diferencias según el tipo de plan, y analizar la **forma de la distribución**.

**Instrucciones:**  
Graficar **histogramas** para las siguientes columnas:  
- `age` (edad de los usuarios)
- `cant_mensajes`
- `cant_llamadas`
- `total_minutos_llamada` 

Después de cada gráfico, escribe un **insight** respecto al plan y la variable, por ejemplo:  
- "Dentro del plan Premium, hay mayor proporción de..."  
- "Los usuarios Básico tienden a hacer ... llamadas y enviar ... mensajes."  o "No existe algún patrón."
- ¿Qué tipo de distribución tiene ? (simétrica, sesgada a la derecha o a la izquierda) 

**Hint**  
Para cada histograma, 
- Usa `hue='plan'` para ver cómo varían las distribuciones según el plan (Básico o Premium).
- Usa `palette=['skyblue','green']`
- Agrega título y etiquetas

In [ ]:
# Histograma para visualizar la edad (age)
sns.histplot(
    data=user_profile,
    x='age',
    hue='plan',
    bins=10,
    palette=['skyblue', 'green']
)

plt.title('Distribución de edad por tipo de plan')
plt.xlabel('Edad')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡Insights: 
- Distribución: la edad de los usuarios se distribuye aproximadamente entre 18 y 79 años. No se observa una diferencia fuerte entre los planes Básico y Premium, ya que ambos aparecen en todos los rangos de edad. El plan Básico tiene más usuarios en casi todos los rangos, lo cual es consistente con que es el plan más frecuente del dataset.

In [ ]:
# Histograma para visualizar la cant_mensajes
sns.histplot(
    data=user_profile,
    x='cant_mensajes',
    hue='plan',
    bins=10,
    palette=['skyblue', 'green']
)

plt.title('Distribución de cantidad de mensajes por tipo de plan')
plt.xlabel('Cantidad de mensajes')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡Insights: 
- Distribución: la cantidad de mensajes se concentra principalmente entre 3 y 8 mensajes por usuario. El plan Básico tiene mayor cantidad de usuarios en casi todos los rangos, pero ambos planes muestran un comportamiento similar. Se observan pocos usuarios con más de 10 mensajes, por lo que podrían revisarse como casos de mayor uso.

In [ ]:
# Histograma para visualizar la cant_llamadas
sns.histplot(
    data=user_profile,
    x='cant_llamadas',
    hue='plan',
    bins=10,
    palette=['skyblue', 'green']
)

plt.title('Distribución de cantidad de llamadas por tipo de plan')
plt.xlabel('Cantidad de llamadas')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡Insights: 
- Distribución: la cantidad de llamadas se concentra principalmente entre 3 y 7 llamadas por usuario. El plan Básico tiene mayor cantidad de usuarios en la mayoría de rangos, pero ambos planes muestran un patrón similar. Hay pocos usuarios con más de 10 llamadas, por lo que pueden revisarse como casos de mayor uso.

In [ ]:
# Histograma para visualizar la cant_minutos_llamada
sns.histplot(
    data=user_profile,
    x='cant_minutos_llamada',
    hue='plan',
    bins=10,
    palette=['skyblue', 'green']
)

plt.title('Distribución de minutos de llamada por tipo de plan')
plt.xlabel('Minutos de llamada')
plt.ylabel('Cantidad de usuarios')
plt.show()

💡Insights: 
- Distribución: los minutos de llamada presentan una distribución sesgada a la derecha. La mayoría de usuarios se concentra entre 0 y 40 minutos, pero existen algunos casos con consumos mucho más altos, llegando cerca de 150 minutos. Estos valores altos pueden representar usuarios de alto consumo o posibles outliers que conviene revisar más adelante.

### 5.2 Identificación de Outliers

🎯 **Objetivo:**  
Detectar valores extremos en las variables clave de **uso** y **clientes** que podrían afectar el análisis, y decidir si requieren limpieza o revisión adicional.

**Instrucciones:**  
- Usa **boxplots** para identificar visualmente outliers en las siguientes columnas:  
  - `age` 
  - `cant_mensajes`
  - `cant_llamadas`
  - `total_minutos_llamada`  
- Crea un **for** para generar los 4 boxplots automáticamente.
<br>

- Después de crear los gráfico, responde si **existen o no outliers** en las variables.  
- Si hay outliers, crea otro bucle para calcular los límites de esas columnas usando el **método IQR** y decide qué hacer con ellos.
  - Si solamente hay outliers de un solo lado, no es necesario calcular ambos límites.

**Hint:**
- Dentro del bucle, usa `plt.title(f'Boxplot: {col}')` para que el título cambie acorde a la columna.

In [ ]:
# Visualizando usando BoxPlot
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_numericas:
    sns.boxplot(
        data=user_profile,
        x=col,
        color='skyblue'
    )
    plt.title(f'Boxplot: {col}')
    plt.xlabel(col)
    plt.show()

💡Insights: 

- Age: no presenta outliers visibles. La edad se mantiene dentro de un rango lógico, aproximadamente entre 18 y 79 años.
- cant_mensajes: presenta algunos outliers hacia la derecha. La mayoría de usuarios se concentra entre 4 y 7 mensajes, pero existen pocos casos con más de 11 mensajes.
- cant_llamadas: presenta algunos outliers hacia la derecha. La mayoría de usuarios se concentra entre 3 y 6 llamadas, pero hay pocos casos con más de 10 llamadas.
- cant_minutos_llamada: presenta varios outliers hacia la derecha. La mayoría de usuarios se concentra en valores bajos o moderados, pero hay usuarios con consumos superiores a 60 minutos e incluso cercanos a 150 minutos. Estos casos deben revisarse como usuarios de alto consumo o posibles valores atípicos.

In [ ]:
# Calcular límites con el método IQR
columnas_limites = ['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_limites:
    Q1 = user_profile[col].quantile(0.25)
    Q3 = user_profile[col].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    print(col)
    print("Límite inferior:", limite_inferior)
    print("Límite superior:", limite_superior)
    print("Máximo:", user_profile[col].max())
    print()

In [ ]:
# Revisa los límites superiores y el max, para tomar la decisión de mantener los outliers o no
user_profile[columnas_limites].describe()

💡Insights: 

- cant_mensajes: se recomienda mantener los outliers. El límite superior por IQR es 11.5 y el máximo es 17, por lo que son valores altos pero no imposibles para el negocio. Pueden representar usuarios con mayor actividad de mensajes.
- cant_llamadas: se recomienda mantener los outliers. El límite superior por IQR es 10.5 y el máximo es 15, lo cual puede representar usuarios con mayor uso de llamadas, no necesariamente errores.
- cant_minutos_llamada: se recomienda mantener los outliers, pero analizarlos como segmento de alto consumo. El límite superior por IQR es 61.86 y el máximo es 155.69, por lo que existen usuarios con consumo de llamadas mucho mayor al típico. Estos casos pueden ser clientes intensivos o posibles comportamientos atípicos que conviene revisar.

---

## 🧩Paso 6: Segmentación de Clientes

### 6.1 Segmentación de Clientes Por Uso

🎯 **Objetivo:** Clasificar a cada usuario en un grupo de uso (Bajo uso, Uso medio, Alto uso) basándose en la cantidad de llamadas y mensajes registrados.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_uso` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones de llamadas y mensajes y asigna:
  - `'Bajo uso'` cuando llamadas < 5 y mensajes < 5
  - `'Uso medio'` cuando llamadas < 10 y mensajes < 10
  - `'Alto uso'` para el resto de casos

In [ ]:
# Crear columna grupo_uso
user_profile['grupo_uso'] = np.where(
    (user_profile['cant_llamadas'] < 5) & (user_profile['cant_mensajes'] < 5),
    'Bajo uso',
    np.where(
        (user_profile['cant_llamadas'] < 10) & (user_profile['cant_mensajes'] < 10),
        'Uso medio',
        'Alto uso'
    )
)

In [ ]:
# verificar cambios
user_profile.head()

### 6.2 Segmentación de Clientes Por Edad

🎯 **Objetivo:**: Clasificar a cada usuario en un grupo por **edad**.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_edad` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones y asigna:
  - `'Joven'` cuando age < 30
  - `'Adulto'` cuando age < 60
  - `'Adulto Mayor'` para el resto de casos

In [ ]:
user_profile['grupo_edad'] = np.select(
    [user_profile['age'].isna(), user_profile['age'] < 30, user_profile['age'] < 60],
    ['Sin dato', 'Joven', 'Adulto'], default='Adulto Mayor')


In [ ]:
# verificar cambios
user_profile.head()

### 6.3 Visualización de la Segmentación de Clientes

🎯 **Objetivo:** Visualizar la distribución de los usuarios según los grupos creados: **grupo_uso** y **grupo_edad**.

**Instrucciones:**  
- Crea dos gráficos para las variables categóricas `grupo_uso` y `grupo_edad`.
- Agrega título y etiquetas a los ejes en cada gráfico.

In [ ]:
# Visualización de los segmentos por uso
sns.countplot(
    data=user_profile,
    x='grupo_uso'
)

plt.title('Distribución de clientes por grupo de uso')
plt.xlabel('Grupo de uso')
plt.ylabel('Cantidad de usuarios')
plt.show()

In [ ]:
# Visualización de los segmentos por edad
sns.countplot(
    data=user_profile,
    x='grupo_edad'
)

plt.title('Distribución de clientes por grupo de edad')
plt.xlabel('Grupo de edad')
plt.ylabel('Cantidad de usuarios')
plt.show()


---
## 🧩Paso 7: Insight Ejecutivo para Stakeholders

🎯 **Objetivo:** Traducir los hallazgos del análisis en conclusiones accionables para el negocio, enfocadas en segmentación, patrones de uso y oportunidades comerciales.

**Preguntas a responder:** 
- ¿Qué problemas tenían originalmemte los datos?¿Qué porcentaje, o cantidad de filas, de esa columna representaban?


- ¿Qué segmentos de clientes identificaste y cómo se comportan según su edad y nivel de uso?  
- ¿Qué segmentos parecen más valiosos para ConnectaTel y por qué?  
- ¿Qué patrones de uso extremo (outliers) encontraste y qué implican para el negocio?


- ¿Qué recomendaciones harías para mejorar la oferta actual de planes o crear nuevos planes basados en los segmentos y patrones detectados?

✍️ **Escribe aquí tu análisis ejecutivo:**

## 8.1 Análisis ejecutivo

### ⚠️ Problemas detectados en los datos

- En `users`, se detectaron valores faltantes en `city`: 469 registros, equivalentes al 11.72% del total. Además, el valor `"?"` fue identificado como inconsistente y reemplazado por `NaN`, aumentando los faltantes de ciudad a 565 registros.
- En `churn_date`, se encontraron 3534 valores nulos, equivalentes al 88.35%. Estos valores no se consideran error directamente, ya que probablemente representan clientes activos sin fecha de cancelación.
- En `age`, se identificó el valor sentinel `-999`, el cual no representa una edad válida. Este valor fue reemplazado por la mediana de edad.
- En `reg_date`, se encontraron 40 registros con año 2026, fuera del rango del proyecto, ya que el análisis solo considera datos hasta 2024. Estos registros fueron marcados como `NaT`.
- En `usage`, los nulos de `duration` y `length` dependen del tipo de registro: `duration` aplica para llamadas y `length` para mensajes. Por lo tanto, no se imputaron de forma general.

### 🔍 Segmentos por Edad

- Se crearon tres grupos de edad: `Joven`, `Adulto` y `Adulto Mayor`.
- El segmento con mayor cantidad de clientes es `Adulto`, seguido por `Adulto Mayor` y finalmente `Joven`.
- Esto indica que la base principal de clientes de ConnectaTel está concentrada en usuarios adultos, por lo que las estrategias comerciales deberían enfocarse principalmente en este grupo.
- Los clientes jóvenes pueden ser un segmento útil para ofertas digitales, paquetes de datos o promociones de bajo costo.
- Los adultos mayores podrían requerir planes simples, claros y con beneficios enfocados en llamadas.

### 📊 Segmentos por Nivel de Uso

- Se clasificaron los clientes en `Bajo uso`, `Uso medio` y `Alto uso`, usando la cantidad de llamadas y mensajes.
- La mayoría de clientes pertenece al segmento `Uso medio`, lo que indica un comportamiento moderado en el uso de servicios móviles.
- El segmento `Bajo uso` representa clientes con menor actividad, que podrían estar en riesgo de baja interacción con el servicio.
- El segmento `Alto uso` es más pequeño, pero muy relevante para el negocio porque agrupa usuarios con mayor consumo de llamadas y mensajes.

### ➡️ Esto sugiere que ...

- ConnectaTel tiene una base de clientes principalmente adulta y de uso medio.
- Los usuarios de alto consumo pueden representar una oportunidad para planes Premium, beneficios adicionales o campañas de fidelización.
- Los usuarios de bajo uso podrían necesitar ofertas más económicas, planes básicos o incentivos para aumentar su actividad.
- Los valores extremos en minutos de llamada no deben eliminarse automáticamente, ya que pueden representar clientes intensivos con necesidades reales de mayor consumo.

### 💡 Recomendaciones

- Diseñar campañas comerciales diferenciadas por nivel de uso: bajo, medio y alto.
- Crear ofertas especiales para clientes de `Alto uso`, enfocadas en más minutos, más mensajes o beneficios Premium.
- Revisar los clientes con consumos muy altos de minutos de llamada para validar si son clientes intensivos, posibles errores o comportamientos atípicos.
- Imputar o marcar como `"Unknown"` los valores faltantes de `city` para no perder usuarios en análisis geográficos.
- Mantener los nulos de `churn_date` como indicador de clientes activos, en lugar de eliminarlos.
- Excluir o revisar los registros con fechas fuera de rango antes de hacer análisis temporales.

## Reproducibilidad y alcance

Los datos originales del entorno académico no se distribuyen. Las reglas y cifras históricas requieren una nueva ejecución con datos autorizados. Este análisis no demuestra causalidad, rentabilidad ni reducción del abandono; las recomendaciones son hipótesis de negocio.


Repositorio: https://github.com/Sacuervo26/connectatel-telecom-analysis
